# 第11章 自编码器
## Autoencoder — 压缩与重建的无监督学习

**来源：李宏毅《深度学习教程》第11章 | 对应原书第202-211页**

---

## 一、知识地图：全章结构与脉络

```
第11章 自编码器
├── 11.1 自编码器的基本概念
│   ├── 编码器（Encoder）：图片→低维向量
│   ├── 解码器（Decoder）：低维向量→重建图片
│   ├── 训练目标：重建误差最小化
│   └── 瓶颈（Bottleneck）：信息压缩的核心
├── 11.2 为什么需要自编码器？
│   ├── 降维：高维→低维的精简表示
│   ├── 图片的变化是有限的（流形假设）
│   └── 下游任务的预处理
├── 11.3 去噪自编码器
│   ├── 输入：加噪图片，目标：干净图片
│   ├── 2008年已有此概念
│   └── BERT就是去噪文本自编码器
├── 11.4 特征解耦（Feature Disentanglement）
│   ├── 应用：语音转换（柯南的领结变身器）
│   ├── 不需要成对的声音数据
│   └── 交换内容和音色维度
├── 11.5 离散隐表征
│   ├── 二进制向量（0/1特征有无）
│   ├── 独热向量（无监督分类）
│   ├── VQ-VAE：向量量化+码本
│   └── 文字形式的嵌入（摘要）
└── 11.6 自编码器的其他应用
    ├── 作为生成器（→ VAE）
    └── 图像压缩（有损压缩）
```

## 二、自编码器的基本概念

### 2.1 两个网络，一个目标

自编码器由两个网络组成：

**编码器（Encoder）**：图片 → 低维向量（如100×100×3=30,000维 → 100维）

**解码器（Decoder）**：低维向量 → 重建图片

**训练目标**：编码器的输入 ≈ 解码器的输出

$$\mathcal{L} = \|x - \text{Decoder}(\text{Encoder}(x))\|^2$$

> **类比**：就像把一份长报告压缩成三个关键词，然后让人根据这三个关键词还原报告。关键词选得越好（编码器越强），还原的报告越接近原文（重建误差越小）。

### 2.2 瓶颈（Bottleneck）为什么是关键？

如果编码器和解码器没有限制，它们可以直接"学会"恒等映射——输入什么就输出什么，什么都没学到。

所以故意让中间层（编码器的输出/解码器的输入）特别窄：
- 输入：30,000维（100×100 RGB图片）
- 瓶颈：100维
- 输出：30,000维

信息被强制压缩 → 网络被迫提取最本质的特征。这叫做**信息瓶颈原理**。

### 2.3 为什么压缩后还能重建？

**关键洞察**：图片的变化是有限的！

一个3×3的矩阵理论上有 $2^9=512$ 种可能。但"看起来像图片"的矩阵只占其中极小一部分（可能是只有2种模式）。所以9维空间中的图片实际上只分布在2维流形上，完全可以用2维向量表示。

> 编码器做的事情就是"化繁为简"——发现数据中有限的变化模式，用更简洁的方式表达。

In [ ]:
# ============================================
# PyTorch示例1：基础自编码器
# ============================================
import torch
import torch.nn as nn

class Autoencoder(nn.Module):
    """
    基础自编码器
    28x28 (784维) -> 64维瓶颈 -> 28x28 (784维)重建
    """
    def __init__(self, input_dim=784, bottleneck_dim=64):
        super().__init__()
        # 编码器：逐步降维
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, bottleneck_dim),  # 瓶颈！
        )
        
        # 解码器：逐步恢复
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # 输出[0,1]归一化
        )
    
    def forward(self, x):
        encoded = self.encoder(x)      # 压缩到瓶颈
        decoded = self.decoder(encoded) # 重建
        return decoded, encoded
    
    def encode(self, x):
        """仅编码（用于下游任务的特征提取）"""
        return self.encoder(x)

model = Autoencoder()
print(f"压缩比: {784}/{64} = {784/64:.1f}x")
print("训练目标: MSE(decoder(encoder(x)), x)")
print("\n瓶颈=64维 <- 这是信息压缩的核心！")

## 三、去噪自编码器 = BERT的原理

### 3.1 结构

输入：加噪声的图片 $x_{noisy} = x + \epsilon$

目标：重建干净的图片 $x$

训练损失：$\mathcal{L} = \|\text{Decoder}(\text{Encoder}(x_{noisy})) - x\|^2$

### 3.2 与BERT的对应关系

| 去噪自编码器 | BERT |
|------------|------|
| 输入加噪图片 | 输入带[MASK]的句子 |
| 编码器 → 瓶颈 | BERT的Transformer层 |
| 解码器 → 重建 | 线性层 → 预测原词 |
| 目标：还原干净图片 | 目标：预测被遮的词 |

> BERT本质上就是**去噪文本自编码器**——[MASK]就是"噪声"，预测被遮的词就是"去噪"！

### 3.3 历史脉络
- 2006年：Hinton提出用RBM逐层预训练自编码器（当时认为深层网络不能一起训练）
- 2008年：去噪自编码器出现
- 2018年：BERT = 去噪文本自编码器的Transformer版本
- 2012年后：Hinton自己发现RBM其实没必要，直接用BP训练即可

In [ ]:
# ============================================
# PyTorch示例2：去噪自编码器
# ============================================
class DenoisingAutoencoder(nn.Module):
    """
    去噪自编码器
    输入：加噪声的图片
    目标：原始干净图片
    
    与BERT的对应：噪声=[MASK]，去噪=预测被遮的词
    """
    def __init__(self, input_dim=784, bottleneck_dim=64, noise_std=0.3):
        super().__init__()
        self.noise_std = noise_std
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, bottleneck_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        # 加入高斯噪声
        noise = torch.randn_like(x) * self.noise_std
        x_noisy = x + noise
        x_noisy = torch.clamp(x_noisy, 0, 1)
        
        # 编码→解码，目标是原始干净x
        decoded = self.decoder(self.encoder(x_noisy))
        return decoded, x_noisy

print("去噪自编码器已定义。")
print("核心思想：输入加噪版本，重建干净版本。")
print("BERT完全相同——只是把'高斯噪声'换成了'[MASK]'。")

## 四、特征解耦（Feature Disentanglement）

### 4.1 为什么需要解耦？

普通自编码器的瓶颈向量中，所有信息纠缠在一起：
- 图片的色泽、纹理、形状 → 全混在一个向量里
- 语音的内容、说话人音色 → 分不清哪个维度代表什么

**解耦目标**：让向量的不同维度对应不同"语义因素"。
- 前50维 = 说话内容
- 后50维 = 说话人特征

### 4.2 应用：语音转换（柯南的领结变身器）

传统方法：需要A和B念完全相同的句子 → 不现实

解耦方法：
1. 收集大量任意语音，训练特征解耦的自编码器
2. 编码器将A的声音分解为：内容向量 + 音色(A)向量
3. 编码器将B的声音分解为：内容向量 + 音色(B)向量
4. 取A的内容向量 + B的音色向量 → 解码器 → B的声音说A的内容！

**不需要A和B说相同的话，甚至不需要讲同一种语言！**

## 五、离散隐表征与VQ-VAE

### 5.1 嵌入不一定非要是连续向量

- **二进制**：每维=某种特征的有无（0=男生，1=女生；0=不戴眼镜，1=戴眼镜）
- **独热向量**：只有一维是1 → 可能实现无监督分类（如手写数字0-9自动对应10个独热编码）
- **文字**：文章的摘要！编码器输入长文章，输出一段文字摘要，解码器从摘要还原全文

### 5.2 VQ-VAE（向量量化变分自编码器）

核心组件：**码本（Codebook）**——一组可学习的向量（如32个）。

过程：
1. 编码器输出连续向量
2. 与码本中所有向量计算相似度（类似自注意力的Query-Key）
3. 选择最相似的向量作为解码器输入

解码器的输入只有32种可能 → 实现了**离散化**的隐表征。

在语音应用中，码本中的向量可能自动学到对应基本发音单元（音素/音标）！

### 5.3 文字嵌入的问题与解决方案

如果让编码器直接输出文字作为嵌入：
- 编码器和解码器会"发明暗号"——产生人类看不懂但解码器能解读的文字
- 解决方案：加GAN的判别器！判别器判断输出是否像人写的自然语言
- 这和Cycle GAN的思路一模一样——Cycle GAN本质就是自编码器+判别器

## 六、自编码器的其他应用

### 6.1 作为生成器

解码器本身就是"输入向量→输出图片" → 可以当生成器用！

从高斯分布采样一个向量 → 输入训练好的解码器 → 生成图片。

**VAE（变分自编码器）** 就是在这个基础上发展出来的——在第8章中提到过，是除了GAN之外的另一种重要生成模型。

### 6.2 图像压缩

- 编码器 = 压缩（图片→小向量）
- 解码器 = 解压缩（小向量→图片）

与JPEG的比较：都是**有损压缩**（因为重建不可能100%完美）。

优势：自编码器是针对特定数据域训练的自适应压缩——如果只压缩人脸图片，它可以学到比通用JPEG更好的压缩效率。

## 七、跨章节连接

| 章节 | 连接关系 |
|------|----------|
| Ch8 GAN | VAE是自编码器+生成模型。Cycle GAN本质就是自编码器。判别器可加入自编码器实现文字嵌入 |
| Ch9 扩散模型 | 去噪自编码器=扩散模型的去噪模块的前身。两者都从噪声中恢复信号 |
| Ch10 BERT | BERT = 去噪文本自编码器。编码器+线性解码器 |
| Ch13 迁移学习 | 自编码器预训练 → 编码器用作下游任务的特征提取器 |
| Ch14 强化学习 | 文字嵌入自编码器的解码器输出离散文字 → 需要RL方法训练 |

## 八、核心要点总结

1. **自编码器结构**：编码器压缩 + 瓶颈 + 解码器重建。训练目标=重建误差最小化。

2. **瓶颈原理**：压缩迫使网络提取最本质的特征。图片变化有限（流形假设），所以大幅压缩是可能的。

3. **去噪自编码器=BERT原型**：输入加噪版本，重建干净版本。BERT的[MASK]就是噪声。

4. **特征解耦**：让瓶颈向量的不同维度对应不同语义因素。应用：无配对语音转换（柯南的领结变身器）。

5. **离散隐表征**：瓶颈不一定是连续向量。可以是二进制、独热向量、甚至是一段文字。

6. **VQ-VAE**：用码本将连续编码量化为离散编码。在语音中码本可学出音素。

7. **编码器+判别器**：文字形式的嵌入会"发明暗号"，需要GAN来约束输出像自然语言。

8. **解码器=生成器**：解码器天然可作为生成器使用 → VAE由此发展。

9. **图像压缩**：编码器=压缩，解码器=解压缩。自适应有损压缩。

10. **自编码器是自监督学习**：不需要标注数据，输入自己就是监督信号。

## 九、练习与思考

1. 解释为什么自编码器的瓶颈越窄，学到的特征可能越本质？有没有瓶颈太窄反而学不到东西的情况？

2. 证明：如果没有瓶颈（编码器和解码器足够大），自编码器可以退化为恒等映射，什么都学不到。

3. 比较去噪自编码器和BERT的异同。为什么说BERT是一个去噪文本自编码器？

4. 特征解耦在语音转换中解决了什么实际问题？为什么传统方法需要"念相同的句子"？

5. 实现VQ-VAE中的码本查找机制（给定编码向量和码本，找到最近的码本向量）。

6. 为什么训练"输出文字摘要"的自编码器时，编码器和解码器会"发明暗号"？这和Cycle GAN有什么联系？

7. 将训练好的自编码器的解码器作为生成器使用，会有什么问题？（提示：输入空间的哪些区域对应"有意义"的图片？）

8. 自编码器用于图像压缩和JPEG压缩有什么不同？两者的"失真"有什么本质区别？